# Time buckets

### Variables

In [1]:
source = '/tmp/warehouse_oa/*.parquet' # where the query starts
bucket_size_minutes = 5

### Setup

In [2]:

## general setup
import sys
import duckdb
import warnings
from functools import reduce

sys.path.append(".")
warnings.filterwarnings('ignore')

# generate snippets
queryZones=f"""SELECT DISTINCT 
  split_part(event_name, 'zone_', 2) as zone
FROM '{source}'
ORDER BY zone;"""
zones = duckdb.sql(queryZones).pl().get_column('zone')

queryData = f"""
WITH base_events AS (
  SELECT 
    track_id,
    event_ts,
    DATE_TRUNC('minute', event_ts) - 
      (INTERVAL '{bucket_size_minutes} minute' * (EXTRACT(MINUTE FROM event_ts)::int % {bucket_size_minutes})) as time_bucket,
    event_name,
    REGEXP_REPLACE(event_name, '^(enter|leave)_zone_', '') as zone_name,
    CASE 
      WHEN event_name LIKE 'enter%' THEN 'enter'
      WHEN event_name LIKE 'leave%' THEN 'leave'
    END as action_type
  FROM '{source}'
  WHERE event_name LIKE '%zone_%'
),

zone_pairs AS (
  SELECT 
    e.track_id,
    e.zone_name,
    e.event_ts as enter_time,
    MIN(l.event_ts) as leave_time
  FROM base_events e
  LEFT JOIN base_events l ON 
    e.track_id = l.track_id AND
    e.zone_name = l.zone_name AND
    l.action_type = 'leave' AND
    l.event_ts > e.event_ts
  WHERE e.action_type = 'enter'
  GROUP BY 
    e.track_id,
    e.zone_name,
    e.event_ts
),

time_buckets AS (
  SELECT time_bucket
  FROM (
    SELECT DISTINCT DATE_TRUNC('minute', event_ts) - 
      (INTERVAL '{bucket_size_minutes} minute' * (EXTRACT(MINUTE FROM event_ts)::int % {bucket_size_minutes})) as time_bucket
    FROM base_events
  ) t
),

bucket_durations AS (
  SELECT 
    zp.track_id,
    zp.zone_name,
    tb.time_bucket,
    -- Calculate the intersection of the visit with the bucket
    GREATEST(zp.enter_time, tb.time_bucket) as bucket_start,
    LEAST(zp.leave_time, tb.time_bucket + INTERVAL '{bucket_size_minutes} minutes') as bucket_end,
    zp.enter_time,
    zp.leave_time
  FROM zone_pairs zp
  CROSS JOIN time_buckets tb
  WHERE zp.leave_time IS NOT NULL
    AND tb.time_bucket <= zp.leave_time 
    AND tb.time_bucket + INTERVAL '{bucket_size_minutes} minutes' > zp.enter_time
),

final_durations AS (
  SELECT 
    track_id,
    zone_name,
    time_bucket,
    -- Calculate the actual duration within this bucket
    EXTRACT(EPOCH FROM (bucket_end - bucket_start)) as duration_seconds
  FROM bucket_durations
  WHERE bucket_end > bucket_start
),

aggregated_data AS (SELECT 
  time_bucket,
  zone_name,
  ROUND(AVG(duration_seconds), 2) as avg_duration_seconds,
  ROUND(MIN(duration_seconds), 2) as min_duration_seconds,
  ROUND(MAX(duration_seconds), 2) as max_duration_seconds,
  COUNT(DISTINCT track_id) as total_visitors,
  COUNT(*) as total_visits
FROM final_durations
GROUP BY 
  time_bucket,
  zone_name
ORDER BY 
  time_bucket,
  zone_name)
  
SELECT 
  time_bucket,
  {",\n".join(map(lambda zone: f"""
  MAX(CASE WHEN zone_name = '{zone}' THEN total_visits ELSE 0 END) AS {zone}_total_visits,
  MAX(CASE WHEN zone_name = '{zone}' THEN total_visitors ELSE 0 END) AS {zone}_total_visitors,
  MAX(CASE WHEN zone_name = '{zone}' THEN avg_duration_seconds ELSE 0 END) AS {zone}_avg_duration_seconds,
  SUM((CASE WHEN zone_name = '{zone}' THEN avg_duration_seconds ELSE 0 END) * (CASE WHEN zone_name = '{zone}' THEN total_visits ELSE 0 END)) AS {zone}_total_time
""", zones))}
FROM aggregated_data
GROUP BY time_bucket
ORDER BY time_bucket
"""

data = duckdb.sql(queryData).df()


### Main

In [3]:
## plot
from plot import plot2
from bokeh.io import output_notebook

output_notebook()
plot2(data, zones, 'total_visitors', bucket_size_minutes)
plot2(data, zones, 'avg_duration_seconds', bucket_size_minutes)
plot2(data, zones, 'total_time', bucket_size_minutes)


Loading BokehJS ...

In [4]:
from IPython.display import display, HTML

display(HTML(data.to_html()))


,time_bucket,center_total_visits,center_total_visitors,center_avg_duration_seconds,center_total_time,entrance_total_visits,entrance_total_visitors,entrance_avg_duration_seconds,entrance_total_time,exit_total_visits,exit_total_visitors,exit_avg_duration_seconds,exit_total_time
0,2025-01-09 16:36:00,20,20,9.44,188.80,9,9,27.80,250.20,17,17,9.67,164.39
1,2025-01-09 16:37:00,28,28,13.34,373.52,11,11,30.10,331.10,20,20,8.73,174.60
2,2025-01-09 16:38:00,34,34,17.61,598.74,17,17,26.84,456.28,22,22,7.97,175.34
3,2025-01-09 16:39:00,39,39,17.32,675.48,20,20,24.96,499.20,26,26,8.00,208.00
4,2025-01-09 16:40:00,36,36,20.26,729.36,22,22,17.65,388.30,19,19,3.66,69.54
5,2025-01-09 16:41:00,31,31,18.64,577.84,16,16,16.23,259.68,14,14,3.56,49.84
6,2025-01-09 16:42:00,27,27,15.78,426.06,22,22,12.25,269.50,9,9,4.79,43.11
7,2025-01-09 16:43:00,29,29,8.71,252.59,20,20,8.28,165.60,12,12,4.81,57.72
8,2025-01-09 16:44:00,28,26,10.24,286.72,18,17,12.32,221.76,13,13,4.21,54.73
9,2025-01-09 16:45:00,37,33,9.96,368.52,18,17,16.00,288.00,24,22,4.14,99.36


In [5]:
import duckdb
from bokeh.plotting import figure, show
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.palettes import Spectral11
from bokeh.transform import linear_cmap

# Fixed transition query
transition_query = f"""
WITH ordered_events AS (
  SELECT 
    track_id,
    event_ts,
    REGEXP_REPLACE(event_name, '^enter_zone_', '') as zone_name,
    LEAD(REGEXP_REPLACE(event_name, '^enter_zone_', '')) OVER (
      PARTITION BY track_id ORDER BY event_ts
    ) as next_zone,
    DATE_TRUNC('minute', event_ts) - 
      (INTERVAL '{bucket_size_minutes} minute' * (EXTRACT(MINUTE FROM event_ts)::int % {bucket_size_minutes})) as time_bucket
  FROM '{source}'
  WHERE event_name LIKE 'enter_zone_%'
)
SELECT 
  time_bucket,
  zone_name as from_zone,
  next_zone as to_zone,
  COUNT(*) as transition_count
FROM ordered_events
WHERE next_zone IS NOT NULL
GROUP BY time_bucket, from_zone, to_zone
ORDER BY time_bucket, from_zone, to_zone
"""

transitions = duckdb.sql(transition_query).df()

# Fixed percentiles query
percentiles_query = f"""
WITH enter_events AS (
  SELECT 
    track_id,
    REGEXP_REPLACE(event_name, '^enter_zone_', '') as zone_name,
    event_ts as enter_time,
    DATE_TRUNC('minute', event_ts) - 
      (INTERVAL '{bucket_size_minutes} minute' * (EXTRACT(MINUTE FROM event_ts)::int % {bucket_size_minutes})) as time_bucket
  FROM '{source}'
  WHERE event_name LIKE 'enter_zone_%'
),
leave_events AS (
  SELECT 
    track_id,
    REGEXP_REPLACE(event_name, '^leave_zone_', '') as zone_name,
    event_ts as leave_time
  FROM '{source}'
  WHERE event_name LIKE 'leave_zone_%'
),
visit_durations AS (
  SELECT 
    e.track_id,
    e.zone_name,
    e.time_bucket,
    e.enter_time,
    MIN(l.leave_time) as leave_time
  FROM enter_events e
  LEFT JOIN leave_events l ON 
    e.track_id = l.track_id AND
    e.zone_name = l.zone_name AND
    l.leave_time > e.enter_time
  GROUP BY e.track_id, e.zone_name, e.time_bucket, e.enter_time
)
SELECT 
  time_bucket,
  zone_name,
  PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY EXTRACT(EPOCH FROM (leave_time - enter_time))) as p25_seconds,
  PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY EXTRACT(EPOCH FROM (leave_time - enter_time))) as p50_seconds,
  PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY EXTRACT(EPOCH FROM (leave_time - enter_time))) as p75_seconds
FROM visit_durations
WHERE leave_time IS NOT NULL
GROUP BY time_bucket, zone_name
ORDER BY time_bucket, zone_name
"""

percentiles = duckdb.sql(percentiles_query).df()

def plot_transitions_bokeh(transitions_df, time_bucket):
    bucket_data = transitions_df[transitions_df['time_bucket'] == time_bucket]
    unique_zones = list(set(bucket_data['from_zone'].unique()) | set(bucket_data['to_zone'].unique()))
    max_count = bucket_data['transition_count'].max()
    
    p = figure(title=f'Zone Transitions at {time_bucket}',
               x_range=unique_zones, y_range=unique_zones,
               width=600, height=600)
    
    source = ColumnDataSource(bucket_data)
    mapper = linear_cmap(field_name='transition_count', 
                        palette=Spectral11, 
                        low=0, high=max_count)
    
    p.rect(x='from_zone', y='to_zone', width=1, height=1,
           source=source,
           fill_color=mapper,
           line_color=None)
    
    p.add_tools(HoverTool(tooltips=[
        ('From', '@from_zone'),
        ('To', '@to_zone'),
        ('Count', '@transition_count')
    ]))
    
    p.xaxis.axis_label = 'From Zone'
    p.yaxis.axis_label = 'To Zone'
    p.xgrid.grid_line_color = None
    p.ygrid.grid_line_color = None
    p.xaxis.major_label_orientation = 0.7
    
    return p

def plot_percentiles_bokeh(percentiles_df, zone_name):
    zone_data = percentiles_df[percentiles_df['zone_name'] == zone_name]
    source = ColumnDataSource(zone_data)
    
    p = figure(title=f'Dwell Time Percentiles for {zone_name}',
               x_axis_type='datetime',
               width=800, height=400)
    
    band = p.varea(x='time_bucket', y1='p25_seconds', y2='p75_seconds',
                   source=source, fill_alpha=0.3,
                   legend_label='IQR')
    
    line = p.line(x='time_bucket', y='p50_seconds',
                  source=source, line_width=2,
                  color='navy', legend_label='Median')
    
    p.add_tools(HoverTool(tooltips=[
        ('Time', '@time_bucket{%F %T}'),
        ('25th percentile', '@p25_seconds{0.0}s'),
        ('Median', '@p50_seconds{0.0}s'),
        ('75th percentile', '@p75_seconds{0.0}s')
    ],
    formatters={'@time_bucket': 'datetime'}))
    
    p.xaxis.axis_label = 'Time'
    p.yaxis.axis_label = 'Seconds'
    p.legend.location = 'top_right'
    
    return p

# Display plots
first_bucket = transitions['time_bucket'].iloc[0]
first_zone = zones[0]

from bokeh.palettes import Category10

def plot_transitions_histogram(transitions_df, zones):
    # Combine 'from_zone' and 'to_zone' to create unique transition identifiers
    transitions_df['transition'] = transitions_df['from_zone'] + ' -> ' + transitions_df['to_zone']
    
    # Pivot data for histogram
    pivot_data = transitions_df.pivot_table(
        index='time_bucket',
        columns='transition',
        values='transition_count',
        aggfunc='sum'
    ).fillna(0)
    
    p = figure(title='Zone Transitions Over Time',
               x_axis_type='datetime',
               width=800, height=400)
    
    colors = Category10[10]  # Adjust if more than 10 unique transitions
    transition_names = pivot_data.columns.tolist()
    
    for transition, color in zip(transition_names, colors):
        source = ColumnDataSource({
            'time_bucket': pivot_data.index,
            'count': pivot_data[transition]
        })
        
        p.line('time_bucket', 'count',
               line_color=color,
               legend_label=transition,
               source=source,
               line_width=2)
    
    p.add_tools(HoverTool(tooltips=[
        ('Time', '@time_bucket{%F %T}'),
        ('Transition', '@count')
    ],
    formatters={'@time_bucket': 'datetime'}))
    
    p.xaxis.axis_label = 'Time'
    p.yaxis.axis_label = 'Number of Transitions'
    p.legend.location = 'top_right'
    p.legend.click_policy = 'hide'
    
    return p

# Display plots
show(column(
    plot_transitions_histogram(transitions, zones),
    plot_percentiles_bokeh(percentiles, zones[0])
))
